<a href="https://colab.research.google.com/github/kamilabalabatyr/Data-Science/blob/main/hw7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание 7: структуры данных и функции

**Как устроено.** Две части по примерно 45 минут: **A** — структуры данных и перебор (час 1 урока), **B** — функции и рекурсия (час 2). Бонус **C** — снова Numpy. После каждого пункта — ✅ проверка: запустите её.
- ★ — по образцу из урока, ★★ — соединить две идеи, ★★★ — подумать самому (по желанию).
- Ответы записывайте в переменные с указанными именами, функции — с указанными именами и параметрами: проверки ищут именно их.
- 🟢 **Обязательно:** A1–A8 и B1–B7. Отмеченное «по желанию», B8 и бонус C — для тех, кто хочет глубже.
- **Функции возвращают значения** (`return`), а не печатают их.

**Окна рядом.** У каждого пункта указан слайд урока (🖥, номер — в правом нижнем углу слайда) и шаг рабочей тетради `lesson7_workbook.ipynb` (📓), где та же идея разобрана подробно. Застряли — откройте их.

**Не только «напиши по условию».** Здесь есть: 🔍 найди ошибку в чужом коде, 🧱 граничные случаи, 🧹 грязные данные, 🔄 требования поменялись, 🗣 ответьте своими словами «почему».

**Сдаётся этот файл.** Перед сдачей: «Restart» → «Run All» — всё должно выполниться сверху вниз, а проверки — дать ✅.

---
# Данные: онлайн-кинотеатр

Пятеро друзей две недели смотрят фильмы. `views` — журнал просмотров: каждый просмотр — словарь. `minutes` — сколько минут посмотрели (бросили — меньше длины фильма), `rating` — оценка **от 0 до 5** или `None`, если не оценили. `catalog` — каталог разделов, жанров, сериалов и сезонов (нужен в части B).

Запустите ячейку. Код генерации разбирать не обязательно — но после урока вы его прочитаете.

In [1]:
import numpy as np

users = ["Аня", "Борис", "Вика", "Гоша", "Дана"]
genres = ["фантастика", "комедия", "мультфильм"]
movies = {                      # название → (жанр, длительность в минутах)
    "Дюна": ("фантастика", 155),
    "Интерстеллар": ("фантастика", 169),
    "Марсианин": ("фантастика", 144),
    "Начало": ("фантастика", 148),
    "Солярис": ("фантастика", 167),
    "Один дома": ("комедия", 103),
    "Маска": ("комедия", 101),
    "Паддингтон": ("комедия", 95),
    "Тайна Коко": ("мультфильм", 105),
    "Вверх": ("мультфильм", 96),
    "Зверополис": ("мультфильм", 108),
}


def make_views(seed=7):
    """Журнал просмотров за 14 дней — список словарей."""
    taste = {"Аня": [6, 1, 2], "Борис": [1, 6, 2], "Вика": [1, 2, 6], "Гоша": [4, 4, 1], "Дана": [2, 2, 2]}
    rng = np.random.default_rng(seed)
    titles = [t for t in movies if t != "Солярис"]          # «Солярис» пока никто не смотрел
    views = []
    for day in range(1, 15):
        for user in users:
            for _ in range(int(rng.integers(0, 3))):        # 0–2 просмотра в день
                w = np.array([taste[user][genres.index(movies[t][0])] for t in titles], dtype=float)
                title = str(rng.choice(titles, p=w / w.sum()))
                genre, length = movies[title]
                share = 1.0 if rng.random() < 0.75 else float(rng.uniform(0.05, 0.9))
                rating = int(rng.integers(0, 6)) if rng.random() < 0.7 else None
                views.append({"user": user, "title": title, "genre": genre,
                              "minutes": round(length * share), "rating": rating, "day": day})
    return views


views = make_views()
titles = [v["title"] for v in views]

# каталог — вложенные словари: раздел → жанр или сериал → сезон → серия: минуты
catalog = {
    "Фильмы": {
        "Фантастика": {"Дюна": 155, "Интерстеллар": 169, "Марсианин": 144, "Начало": 148, "Солярис": 167},
        "Комедии": {"Один дома": 103, "Маска": 101, "Паддингтон": 95},
        "Мультфильмы": {"Тайна Коко": 105, "Вверх": 96, "Зверополис": 108},
    },
    "Сериалы": {
        "Шерлок": {
            "Сезон 1": {"s1e1": 88, "s1e2": 89, "s1e3": 89},
            "Сезон 2": {"s2e1": 89, "s2e2": 88, "s2e3": 89},
        },
        "Тед Лассо": {
            "Сезон 1": {"s1e1": 30, "s1e2": 32, "s1e3": 29},
        },
    },
}

print(len(views), "просмотров")
for v in views[:3]:
    print(v)

70 просмотров
{'user': 'Аня', 'title': 'Вверх', 'genre': 'мультфильм', 'minutes': 23, 'rating': 3, 'day': 1}
{'user': 'Аня', 'title': 'Тайна Коко', 'genre': 'мультфильм', 'minutes': 105, 'rating': None, 'day': 1}
{'user': 'Вика', 'title': 'Тайна Коко', 'genre': 'мультфильм', 'minutes': 105, 'rating': 4, 'day': 1}


---
# Часть A. Структуры данных и перебор

## A1 ★ Списки · 🖥 слайд 4 · 📓 шаг 1
`titles` — названия фильмов по порядку просмотров (уже есть).
1. `n_views` — сколько всего просмотров;
2. `last5` — последние 5 названий (срез);
3. `alpha` — все названия по алфавиту, **не меняя** `titles`;
4. `first_dune` — номер (индекс) первого просмотра «Дюны»; `n_dune` — сколько раз её смотрели.

In [2]:
n_views = len(titles)
last5 = titles[-5:]
alpha = sorted(titles)
first_dune = titles.index("Дюна")
n_dune = titles.count("Дюна")

In [3]:
# ✅ Проверка A1 — просто запустите ячейку
try:
    assert titles == [v["title"] for v in views], "titles изменился — нужен sorted(), а не .sort()"
    assert n_views == 70, f"n_views: {n_views}"
    assert last5 == ["Паддингтон", "Вверх", "Вверх", "Дюна", "Интерстеллар"], f"last5: {last5}"
    assert alpha == sorted(titles) and len(alpha) == 70, "alpha — отсортированная копия titles"
    assert first_dune == 6 and n_dune == 9, f"first_dune={first_dune}, n_dune={n_dune}"
    print("✅ A1: всё верно!")
except AssertionError as e:
    print("❌ A1: пока не так —", e)
except Exception as e:
    print(f"❌ A1: ошибка {type(e).__name__}: {e}")

✅ A1: всё верно!


🗣 **Почему так?** Что было бы, если написать `alpha = titles.sort()`? Что окажется в `alpha` и в `titles`?

**Ваш ответ:** sort() сортирует список на месте и возвращает None. Поэтому в alpha окажется None, а titles станет отсортированным, и исходный порядок просмотров потеряется. Чтобы получить копию и не трогать оригинал, нужен sorted(titles).

## A2 ★ Кортежи и распаковка · 🖥 слайд 5 · 📓 шаг 2
1. Распакуйте первый просмотр: `first_user, first_title` — кто и что смотрел (одной строкой из кортежа `(views[0]["user"], views[0]["title"])`).
2. `leader, *others = users` — первый друг и список остальных.
3. `last_rating` — словарь с ключом-кортежем `(зритель, фильм)` → **последняя** поставленная оценка. Просмотры без оценки (`None`) пропускайте.

In [4]:
first_user, first_title = (views[0]["user"], views[0]["title"])
leader, *others = users
last_rating = {}
for v in views:
  if v["rating"] is not None:
    last_rating[(v["user"], v["title"])] = v["rating"]

In [5]:
# ✅ Проверка A2 — просто запустите ячейку
try:
    assert (first_user, first_title) == ("Аня", "Вверх"), f"{first_user!r}, {first_title!r}"
    assert leader == "Аня" and others == ["Борис", "Вика", "Гоша", "Дана"], f"leader={leader!r}, others={others}"
    assert isinstance(last_rating, dict) and all(isinstance(k, tuple) for k in last_rating), "ключи — кортежи (зритель, фильм)"
    assert None not in last_rating.values(), "None не записываем"
    assert len(last_rating) == 31, f"пар с оценкой должно быть 31, у вас {len(last_rating)}"
    assert last_rating[("Аня", "Вверх")] == 0, "оценка 0 — тоже оценка: проверяйте is not None, а не просто if"
    assert last_rating[("Аня", "Интерстеллар")] == 5 and last_rating[("Гоша", "Дюна")] == 3
    print("✅ A2: всё верно!")
except AssertionError as e:
    print("❌ A2: пока не так —", e)
except Exception as e:
    print(f"❌ A2: ошибка {type(e).__name__}: {e}")

✅ A2: всё верно!


## A3 ★ Словари: подсчёт по группам · 🖥 слайд 6 · 📓 шаг 3
1. `minutes_by_user` — зритель → сколько минут посмотрел всего;
2. `views_by_genre` — жанр → сколько просмотров;
3. ★★ `user_genre` — вложенный словарь: зритель → {жанр → минуты}. Жанры, которые зритель не смотрел, можно не добавлять.

In [6]:
minutes_by_user = {}
for v in views:
  minutes_by_user[v["user"]] = minutes_by_user.get(v["user"], 0) + v["minutes"]
views_by_genre = {}
for v in views:
  views_by_genre[v["genre"]] = views_by_genre.get(v["genre"], 0) + 1
user_genre = {}
for v in views:
  inner = user_genre.setdefault(v["user"], {})
  inner[v["genre"]] = inner.get(v["genre"], 0) + v["minutes"]

In [7]:
# ✅ Проверка A3 — просто запустите ячейку
try:
    assert minutes_by_user == {"Аня": 1936, "Борис": 1094, "Вика": 1185, "Гоша": 2042, "Дана": 1364}, f"minutes_by_user: {minutes_by_user}"
    assert views_by_genre == {"фантастика": 33, "комедия": 17, "мультфильм": 20}, f"views_by_genre: {views_by_genre}"
    assert user_genre["Аня"]["фантастика"] == 1382 and user_genre["Борис"]["комедия"] == 640, "user_genre: проверьте вложенный подсчёт"
    assert user_genre["Борис"].get("фантастика", 0) == 0
    print("✅ A3: всё верно!")
except AssertionError as e:
    print("❌ A3: пока не так —", e)
except Exception as e:
    print(f"❌ A3: ошибка {type(e).__name__}: {e}")

✅ A3: всё верно!


## A4 ★ Множества · 🖥 слайд 7 · 📓 шаг 4
`seen` — зритель → множество фильмов, которые он смотрел (уже есть).
1. `common_ab` — **отсортированный список** фильмов, которые смотрели и Аня, и Борис;
2. `only_anya` — отсортированный список: Аня смотрела, Борис — нет;
3. `never_watched` — **множество** фильмов из `movies`, которые не смотрел никто.

In [8]:
seen = {u: {v["title"] for v in views if v["user"] == u} for u in users}

common_ab = sorted(seen["Аня"] & seen["Борис"])
only_anya = sorted(seen["Аня"] - seen["Борис"])
never_watched = set(movies) - set(titles)

In [9]:
# ✅ Проверка A4 — просто запустите ячейку
try:
    assert common_ab == ["Вверх", "Зверополис", "Один дома"], f"common_ab: {common_ab}"
    assert only_anya == ["Дюна", "Интерстеллар", "Марсианин", "Начало", "Тайна Коко"], f"only_anya: {only_anya}"
    assert never_watched == {"Солярис"}, f"never_watched: {never_watched}"
    print("✅ A4: всё верно!")
except AssertionError as e:
    print("❌ A4: пока не так —", e)
except Exception as e:
    print(f"❌ A4: ошибка {type(e).__name__}: {e}")

✅ A4: всё верно!


## A5 ★ 🔍 Найди ошибку: «резервная копия» · 🖥 слайд 8 · 📓 шаг 5
Перед экспериментом коллега сделал копию списков «посмотреть позже» — и всё равно испортил оригинал. Почините, чтобы `watchlists` не изменился, а `backup` — изменился.

In [12]:
import copy
watchlists = {"Аня": ["Дюна"], "Борис": ["Маска"]}
backup = copy.deepcopy(watchlists)
backup["Аня"].append("Солярис")
print(watchlists)

{'Аня': ['Дюна'], 'Борис': ['Маска']}


In [13]:
# ✅ Проверка A5 — просто запустите ячейку
try:
    assert watchlists == {"Аня": ["Дюна"], "Борис": ["Маска"]}, f"оригинал испорчен: {watchlists}"
    assert backup["Аня"] == ["Дюна", "Солярис"], f"backup: {backup}"
    assert backup["Аня"] is not watchlists["Аня"], "внутренние списки всё ещё общие"
    print("✅ A5: всё верно!")
except AssertionError as e:
    print("❌ A5: пока не так —", e)
except Exception as e:
    print(f"❌ A5: ошибка {type(e).__name__}: {e}")

✅ A5: всё верно!


🗣 **Почему так?** Ведь `.copy()` создал новый словарь — `backup is watchlists` даёт `False`. Почему оригинал всё равно изменился?

**Ваш ответ:** .copy() делает поверхностную копию: создаётся новый словарь, но внутри лежат ярлыки на те же самые списки, поэтому backup["Аня"] и watchlists["Аня"] один и тот же список, и append через копию меняет оригинал. deepcopy копирует и вложенные списки.

## A6 ★ Перебор: `enumerate`, `zip` · 🖥 слайд 9 · 📓 шаг 6
1. `leaderboard` — список строк вида `"1. Гоша — 2042 мин"`: зрители по убыванию минут (используйте `minutes_by_user` из A3, `sorted(..., key=..., reverse=True)` и `enumerate`).
2. `daily` — минуты за каждый из 14 дней (список). `growth_days` — в скольких днях минут стало **больше**, чем накануне (`zip(daily, daily[1:])`).

In [14]:
leaderboard = []
ranked = sorted(minutes_by_user.items(), key=lambda kv: kv[1], reverse=True)
for i, (name, mins) in enumerate(ranked, start=1):
  leaderboard.append(f"{i}. {name} — {mins} мин")
daily = [sum(v["minutes"] for v in views if v["day"] == d) for d in range(1, 15)]
growth_days = sum(1 for a, b in zip(daily, daily[1:]) if a<b)

In [15]:
# ✅ Проверка A6 — просто запустите ячейку
try:
    assert leaderboard == ["1. Гоша — 2042 мин", "2. Аня — 1936 мин", "3. Дана — 1364 мин", "4. Вика — 1185 мин", "5. Борис — 1094 мин"], f"leaderboard: {leaderboard}"
    assert len(daily) == 14 and daily[:3] == [732, 1099, 184], f"daily[:3]: {daily[:3]}"
    assert growth_days == 7, f"growth_days: {growth_days}"
    print("✅ A6: всё верно!")
except AssertionError as e:
    print("❌ A6: пока не так —", e)
except Exception as e:
    print(f"❌ A6: ошибка {type(e).__name__}: {e}")

✅ A6: всё верно!


## A7 ★★ Включения · 🖥 слайд 10 · 📓 шаг 7
1. `avg_rating` — словарь «фильм → средняя оценка», округлённая до 2 знаков, **без** `None`. Фильмы без единой оценки в словарь не попадают. Осторожно: оценка 0 — настоящая оценка.
2. `long_views` — список пар `(зритель, фильм)` для просмотров от 150 минут и дольше, по порядку журнала.
3. `genres_of` — словарь «зритель → множество жанров, которые он смотрел».

In [16]:
ratings = {}
for v in views:
  if v["rating"] is not None:
    ratings.setdefault(v["title"], []).append(v["rating"])
avg_rating = {t: round(sum(r) / len(r), 2) for t, r in ratings.items()}
long_views = [(v["user"], v["title"]) for v in views if v["minutes"]>=150]
genres_of = {u: {v["genre"] for v in views if v["user"]==u} for u in users}

In [17]:
# ✅ Проверка A7 — просто запустите ячейку
try:
    assert "Солярис" not in avg_rating, "фильмы без оценок не добавляем"
    assert avg_rating["Вверх"] == 2.0, f"Вверх: {avg_rating.get('Вверх')} — нули выкинуты? Нужно `is not None`, а не `if v['rating']`"
    assert avg_rating["Маска"] == 5.0 and avg_rating["Паддингтон"] == 3.6 and avg_rating["Марсианин"] == 2.7, f"avg_rating: {avg_rating}"
    assert len(long_views) == 13 and long_views[:2] == [("Дана", "Дюна"), ("Аня", "Интерстеллар")], f"long_views[:2]: {long_views[:2]}"
    assert genres_of["Борис"] == {"комедия", "мультфильм"} and genres_of["Гоша"] == {"фантастика", "комедия"}, f"genres_of: {genres_of}"
    print("✅ A7: всё верно!")
except AssertionError as e:
    print("❌ A7: пока не так —", e)
except Exception as e:
    print(f"❌ A7: ошибка {type(e).__name__}: {e}")

✅ A7: всё верно!


## A8 ★★ 🧹 Генераторы и `Counter` · 🖥 слайд 11 · 📓 шаг 8
1. `top3_titles` — три самых просматриваемых фильма с числом просмотров (`Counter`).
2. `has_zero` — была ли хоть одна оценка 0 (`any`).
3. `n_unrated` — сколько просмотров без оценки.
4. `first_abandoned` — кортеж `(зритель, фильм, день)` первого просмотра, где посмотрели **меньше 30%** фильма (`next`; длина фильма — `movies[название][1]`).

In [19]:
from collections import Counter

top3_titles = Counter(titles).most_common(3)
has_zero = any(v["rating"]==0 for v in views)
n_unrated = sum(1 for v in views if v["rating"] is None)
first_abandoned = next((v["user"], v["title"], v["day"]) for v in views if v["minutes"] < 0.3 * movies[v["title"]][1])

In [20]:
# ✅ Проверка A8 — просто запустите ячейку
try:
    assert top3_titles == [("Марсианин", 11), ("Вверх", 10), ("Дюна", 9)], f"top3_titles: {top3_titles}"
    assert has_zero is True, f"has_zero: {has_zero!r}"
    assert n_unrated == 19, f"n_unrated: {n_unrated}"
    assert first_abandoned == ("Аня", "Вверх", 1), f"first_abandoned: {first_abandoned}"
    print("✅ A8: всё верно!")
except AssertionError as e:
    print("❌ A8: пока не так —", e)
except Exception as e:
    print(f"❌ A8: ошибка {type(e).__name__}: {e}")

✅ A8: всё верно!


🗣 **Почему так?** Коллега посчитал «сколько просмотров с оценкой» как `sum(1 for v in views if v["rating"])` и получил 43, а не 51. Куда делись 8 просмотров?

**Ваш ответ:** if v["rating"] считает 0 ложью, поэтому 8 просмотров с оценкой 0 выпали из подсчёта вместе с None. Правильно писать if v["rating"] is not None.

---
# Часть B. Функции и рекурсия

## B1 ★ Параметры по имени · 🖥 слайд 15 · 📓 шаг F1
`watch_time(user, *, genre=None, min_rating=None)` — сколько минут `user` посмотрел. Если задан `genre` — только этот жанр; если задан `min_rating` — только просмотры с оценкой **не ниже** этой (без оценки не считаются). `genre` и `min_rating` — **только по имени**.

In [24]:
def watch_time(user, *, genre=None, min_rating=None):
    total = 0
    for v in views:
        if v["user"] != user:
            continue
        if genre is not None and v["genre"] != genre:
            continue
        if min_rating is not None and (v["rating"] is None or v["rating"] < min_rating):
            continue
        total += v["minutes"]
    return total

In [25]:
# ✅ Проверка B1 — просто запустите ячейку
try:
    assert watch_time("Аня") == 1936, f"watch_time('Аня') = {watch_time('Аня')}"
    assert watch_time("Аня", genre="фантастика") == 1382
    assert watch_time("Аня", min_rating=4) == 722, f"min_rating=4: {watch_time('Аня', min_rating=4)}"
    assert watch_time("Аня", genre="фантастика", min_rating=4) == 619
    assert watch_time("Зарина") == 0, "🧱 нет такого зрителя — 0"
    try:
        watch_time("Аня", "фантастика")
        raise AssertionError("genre должен быть только по имени: поставьте * в сигнатуре")
    except TypeError:
        pass
    print("✅ B1: всё верно!")
except AssertionError as e:
    print("❌ B1: пока не так —", e)
except Exception as e:
    print(f"❌ B1: ошибка {type(e).__name__}: {e}")

✅ B1: всё верно!


## B2 ★ `**kwargs` и `*args` · 🖥 слайд 16 · 📓 шаг F2
1. `filter_views(views, **conditions)` — просмотры, у которых **все** поля из `conditions` равны заданным: `filter_views(views, user="Аня", genre="мультфильм")`.
2. `total_minutes(*names)` — сколько минут посмотрели все перечисленные зрители вместе.

In [26]:
def filter_views(views, **conditions):
    return [v for v in views if all(v.get(k) == val for k, val in conditions.items())]

def total_minutes(*names):
    return sum(v['minutes'] for v in views if v["user"] in names)

In [27]:
# ✅ Проверка B2 — просто запустите ячейку
try:
    assert len(filter_views(views, user="Аня", genre="мультфильм")) == 6
    assert len(filter_views(views, day=3)) == 2
    assert len(filter_views(views)) == 70, "без условий — все просмотры"
    assert total_minutes("Аня", "Борис") == 3030, f"total_minutes('Аня', 'Борис') = {total_minutes('Аня', 'Борис')}"
    assert total_minutes(*users) == sum(v["minutes"] for v in views)
    assert total_minutes() == 0, "🧱 без имён — 0"
    print("✅ B2: всё верно!")
except AssertionError as e:
    print("❌ B2: пока не так —", e)
except Exception as e:
    print(f"❌ B2: ошибка {type(e).__name__}: {e}")

✅ B2: всё верно!


## B3 ★ Вернуть три значения · 🖥 слайд 17 · 📓 шаг F3
`user_stats(user)` возвращает кортеж: число просмотров, сумму минут и среднюю оценку (2 знака, без `None`). Если оценок нет — вместо средней `None`. Распакуйте статистику Вики в `n_vika`, `min_vika`, `avg_vika`.

In [30]:
def user_stats(user):
    mine = [v for v in views if v["user"]==user]
    rated = [v["rating"] for v in mine if v["rating"] is not None]
    avg = round(sum(rated)/len(rated), 2) if rated else None
    return len(mine), sum(v["minutes"] for v in mine), avg

n_vika, min_vika, avg_vika = user_stats("Вика")

In [31]:
# ✅ Проверка B3 — просто запустите ячейку
try:
    assert user_stats("Вика") == (13, 1185, 2.11), f"user_stats('Вика') = {user_stats('Вика')}"
    assert (n_vika, min_vika, avg_vika) == (13, 1185, 2.11)
    assert user_stats("Аня") == (18, 1936, 3.0)
    assert user_stats("Зарина") == (0, 0, None), "🧱 без просмотров — (0, 0, None), без ZeroDivisionError"
    print("✅ B3: всё верно!")
except AssertionError as e:
    print("❌ B3: пока не так —", e)
except Exception as e:
    print(f"❌ B3: ошибка {type(e).__name__}: {e}")

✅ B3: всё верно!


## B4 ★ 🔍 Найди ошибку: список по умолчанию · 🖥 слайд 18 · 📓 шаг F4
У каждого вызова без `watchlist` должен быть свой новый список. Почините.

In [32]:
def add_to_watchlist(title, watchlist=None):
  if watchlist is None:
    watchlist = []
  watchlist.append(title)
  return watchlist

print(add_to_watchlist("Дюна"))
print(add_to_watchlist("Маска"))

['Дюна']
['Маска']


In [33]:
# ✅ Проверка B4 — просто запустите ячейку
try:
    assert add_to_watchlist("Вверх") == ["Вверх"]
    assert add_to_watchlist("Начало") == ["Начало"], "второй вызов видит фильмы первого — список по умолчанию общий"
    mine = ["Дюна"]
    assert add_to_watchlist("Солярис", mine) == ["Дюна", "Солярис"] and mine == ["Дюна", "Солярис"]
    print("✅ B4: всё верно!")
except AssertionError as e:
    print("❌ B4: пока не так —", e)
except Exception as e:
    print(f"❌ B4: ошибка {type(e).__name__}: {e}")

✅ B4: всё верно!


## B5 ★ 🔍 Найди ошибку: `print` и внешняя переменная · 🖥 слайды 17–18 · 📓 шаги F3–F4
Функция должна **возвращать** минуты зрителя, но падает. Почините и соберите `all_minutes` — словарь «зритель → минуты» через `count_minutes`.

In [34]:
total = 0

def count_minutes(user):
    total = 0
    for v in views:
        if v["user"] == user:
            total += v["minutes"]
    return total

all_minutes = {u: count_minutes(u) for u in users}

In [35]:
# ✅ Проверка B5 — просто запустите ячейку
try:
    assert count_minutes("Аня") == 1936, "функция должна вернуть число (return), а не напечатать"
    assert count_minutes("Аня") == 1936, "второй вызов дал другое число — сумма копится между вызовами?"
    assert all_minutes == minutes_by_user, f"all_minutes: {all_minutes}"
    print("✅ B5: всё верно!")
except AssertionError as e:
    print("❌ B5: пока не так —", e)
except Exception as e:
    print(f"❌ B5: ошибка {type(e).__name__}: {e}")

✅ B5: всё верно!


🗣 **Почему так?** Почему исходный код упал с `UnboundLocalError`, хотя `total = 0` написано прямо над функцией?

**Ваш ответ:** внутри функции есть присваивание total += …, поэтому Python считает total локальной переменной этой функции. Внешний total = 0 она не видит, а локальный ещё не создан, и прибавлять не к чему.

## B6 ★★ `lambda` и сортировка по двум ключам · 🖥 слайд 19 · 📓 шаг F5
1. `user_rank` — имена зрителей по убыванию минут (`minutes_by_user`), при равенстве — по алфавиту.
2. `best_titles` — три фильма с самой высокой средней оценкой (`avg_rating` из A7) среди тех, у кого **не меньше 3 оценок**; при равенстве — по алфавиту. Список названий.

In [36]:
user_rank = sorted(minutes_by_user, key=lambda u: (-minutes_by_user[u], u))
n_ratings = {t: sum(1 for v in views if v["title"] == t and v["rating"] is not None) for t in movies}
best_titles = sorted([t for t in avg_rating if n_ratings[t] >= 3], key=lambda t: (-avg_rating[t], t))[:3]

In [37]:
# ✅ Проверка B6 — просто запустите ячейку
try:
    assert user_rank == ["Гоша", "Аня", "Дана", "Вика", "Борис"], f"user_rank: {user_rank}"
    assert best_titles == ["Паддингтон", "Один дома", "Начало"], f"best_titles: {best_titles} — «Маска» с одной оценкой не считается"
    print("✅ B6: всё верно!")
except AssertionError as e:
    print("❌ B6: пока не так —", e)
except Exception as e:
    print(f"❌ B6: ошибка {type(e).__name__}: {e}")

✅ B6: всё верно!


## B7 ★★ Рекурсия по каталогу · 🖥 слайды 20–21 · 📓 шаги F6–F7
`catalog` — вложенные словари: раздел → жанр или сериал → сезон → серия. Листья — длительность в минутах.
1. `count_titles(node)` — сколько всего листьев (фильмов и серий);
2. `total_runtime(node)` — сумма минут всех листьев;
3. `find_path(node, target)` — путь к первому листу с именем `target` вида `"Сериалы / Шерлок / Сезон 2 / s2e1"`, или `None`, если такого нет.

In [38]:
def count_titles(node):
    total = 0
    for name, value in node.items():
        if isinstance(value, dict):
            total += count_titles(value)
        else:
            total += 1
    return total

def total_runtime(node):
    total = 0
    for name, value in node.items():
        if isinstance(value, dict):
            total += total_runtime(value)
        else:
            total += value
    return total

def find_path(node, target, path=""):
    for name, value in node.items():
        if isinstance(value, dict):
            found = find_path(value, target, path + name + " / ")
            if found is not None:
                return found
        elif name == target:
            return path + name
    return None

In [39]:
# ✅ Проверка B7 — просто запустите ячейку
try:
    assert count_titles(catalog) == 20, f"count_titles: {count_titles(catalog)}"
    assert total_runtime(catalog) == 2014, f"total_runtime: {total_runtime(catalog)}"
    assert total_runtime(catalog["Сериалы"]["Тед Лассо"]) == 91, "функция работает с любым поддеревом"
    assert find_path(catalog, "s2e1") == "Сериалы / Шерлок / Сезон 2 / s2e1", f"find_path: {find_path(catalog, 's2e1')!r}"
    assert find_path(catalog, "s1e1") == "Сериалы / Шерлок / Сезон 1 / s1e1", "первый найденный — у Шерлока"
    assert find_path(catalog, "Вверх") == "Фильмы / Мультфильмы / Вверх"
    assert find_path(catalog, "Титаник") is None, "🧱 не нашли — None"
    print("✅ B7: всё верно!")
except AssertionError as e:
    print("❌ B7: пока не так —", e)
except Exception as e:
    print(f"❌ B7: ошибка {type(e).__name__}: {e}")

✅ B7: всё верно!


🗣 **Почему так?** Почему здесь не обойтись двумя вложенными циклами `for`, как в подсчёте по дням? Что изменится, если у «Шерлока» появятся сезоны с «дополнительными материалами» внутри?

**Ваш ответ:** у каталога разная глубина. У фильмов 3 уровня, у сериалов 4. Число циклов пришлось бы угадывать заранее, а если появятся «доп. материалы» внутри сезона, код сломается. Рекурсия спускается на любую глубину сама.